# Apache Hudi Core Conceptions (3) - MOR: File Layouts + Compaction

## 1. Configuration

In [1]:
%%configure -f
{
    "conf" : {
        "spark.jars":"hdfs:///tmp/hudi-spark-bundle.jar",            
        "spark.serializer":"org.apache.spark.serializer.KryoSerializer",
        "spark.sql.extensions":"org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
51,application_1678096020253_0074,spark,idle,Link,Link,None,


In [2]:
%%sh
# deploy hudi bundle jar
hdfs dfs -copyFromLocal -f /usr/lib/hudi/hudi-spark-bundle.jar /tmp/hudi-spark-bundle.jar
hdfs dfs -ls /tmp/hudi-spark-bundle.jar
# deploy hudi-stat.sh - a utility shell script 
wget https://github.com/bluishglc/hudi-core-conceptions/releases/download/v1.0/hudi-stat.sh -O ~/hudi-stat.sh &>/dev/null
chmod a+x ~/hudi-stat.sh
ls ~/hudi-stat.sh

-rw-r--r--   1 emr-notebook hdfsadmingroup   61421977 2023-03-10 10:17 /tmp/hudi-spark-bundle.jar
/home/emr-notebook/hudi-stat.sh


In [3]:
%%html
<style>
table {float:left}
</style>

## 2. Test Case 1 - Sync Compaction: Inline Schedule + Inline Execute

### 2.1. Test Plan

Step No.|Action|Volume Per Partition |Storage
:--------:|:------|:------|:----------
1|Insert|41MB|+1 Base File
2|Update|177KB|+1 Log File
3|Update|252KB|+1 Log File, +1 Base File
4|Update|258KB|+1 Log File
5|Update|280KB|+1 Log File

### 2.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.compact.inline|false|true
hoodie.compact.schedule.inline|false|Default Value
hoodie.compact.inline.max.delta.commits|5|3
hoodie.copyonwrite.record.size.estimate|1024|175

### 2.3. Set Variables

In [4]:
%%sql
set TABLE_NAME=reviews_mor_1

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
52,application_1678096020253_0075,spark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [5]:
%env TABLE_NAME=reviews_mor_1

env: TABLE_NAME=reviews_mor_1


### 2.4. Create Table

In [6]:
%%sh
aws s3 rm s3://${S3_BUCKET}/${TABLE_NAME} --recursive &>/dev/null
rm -rf ${WORKSPACE}/${TABLE_NAME}
sleep 5

In [7]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [8]:
%%sql
drop table if exists ${TABLE_NAME}_ro

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [9]:
%%sql
drop table if exists ${TABLE_NAME}_rt

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [10]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    hoodie.compact.inline = 'true',
    -- hoodie.compact.schedule.inline = 'false',
    hoodie.compact.inline.max.delta.commits = '3'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
java.lang.IllegalArgumentException: Can not create a Path from an empty string
  at org.apache.hadoop.fs.Path.checkPathArg(Path.java:172)
  at org.apache.hadoop.fs.Path.<init>(Path.java:184)
  at org.apache.spark.sql.catalyst.catalog.CatalogUtils$.stringToURI(ExternalCatalogUtils.scala:259)
  at org.apache.spark.sql.catalyst.analysis.ResolveSessionCatalog.$anonfun$getStorageFormatAndProvider$1(ResolveSessionCatalog.scala:464)
  at scala.Option.map(Option.scala:230)
  at org.apache.spark.sql.catalyst.analysis.ResolveSessionCatalog.org$apache$spark$sql$catalyst$analysis$ResolveSessionCatalog$$getStorageFormatAndProvider(ResolveSessionCatalog.scala:464)
  at org.apache.spark.sql.catalyst.analysis.ResolveSessionCatalog$$anonfun$apply$1.applyOrElse(ResolveSessionCatalog.scala:153)
  at org.apache.spark.sql.catalyst.analysis.ResolveSessionCatalog$$anonfun$apply$1.applyOrElse(ResolveSessionCatalog.scala:49)
  at org.apache.spark.sql.catalyst.plans.logical.AnalysisHel

In [11]:
%%sql
set YEAR=1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [12]:
%%sql
select
    l.*, l.records_num / r.total as percentage
from
    (select star_rating, count(1) as records_num from reviews where year=${YEAR} group by star_rating) as l
join
    (select count(1) as total from reviews where year=${YEAR}) as r
order by 
    star_rating asc;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 2.5. Insert 41MB / +1 Base File

In [13]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.sql.AnalysisException: Table not found: reviews_mor_1; line 2 pos 4;
'InsertIntoStatement 'UnresolvedRelation [reviews_mor_1], [], false, false, false
+- Project [review_id#65, star_rating#66, review_body#67, review_date#68, year#69, unix_timestamp(current_timestamp(), yyyy-MM-dd HH:mm:ss, Some(UTC), false) AS timestamp#96L, mod(crc32(cast(review_id#65 as binary)), cast(2 as bigint)) AS parity#97L]
   +- Filter (year#69 = 1999)
      +- SubqueryAlias spark_catalog.default.reviews
         +- Relation default.reviews[review_id#65,star_rating#66,review_body#67,review_date#68,year#69] parquet

  at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.failAnalysis(package.scala:42)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:140)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1$adapted(CheckAnalysis.scala:101)
  at org.apache.spark.sql.catalyst.t

In [14]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ COMMITS ]


[ COMPACTIONS ]


[ STORAGE ]

/home/emr-notebook/timeline
    0 used in 0 directories, 0 files


### 2.6. Update 177KB / +1 Log File

In [15]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.sql.AnalysisException: Table or view not found: reviews_mor_1; line 2 pos 4;
'UpdateTable [assignment('review_body, concat(uuid(Some(-5774938405852095799)), uuid(Some(6899967276471176931)), uuid(Some(1594385811041323352)), uuid(Some(-3969253625396772687)), uuid(Some(8794852617454571427)), uuid(Some(-8267611009338656979)), uuid(Some(-6495577955758210932)), uuid(Some(-1041716749814459685)), uuid(Some(-4467637115816941223)), uuid(Some(-6479292649973075290)))), assignment('timestamp, unix_timestamp(current_timestamp(), yyyy-MM-dd HH:mm:ss, Some(UTC), false))], ('review_date = 1999-01-01)
+- 'UnresolvedRelation [reviews_mor_1], [], false

  at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.failAnalysis(package.scala:42)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:130)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1$adapted(CheckAnalysis.sca

In [16]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ COMMITS ]


[ COMPACTIONS ]


[ STORAGE ]

/home/emr-notebook/timeline
    0 used in 0 directories, 0 files


### 2.7. Update 252KB / +1 Log File +1 Base File

In [17]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-02'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.sql.AnalysisException: Table or view not found: reviews_mor_1; line 2 pos 4;
'UpdateTable [assignment('review_body, concat(uuid(Some(6294790932601769602)), uuid(Some(-148086057980580148)), uuid(Some(-7181195676624854580)), uuid(Some(-3121103696469405927)), uuid(Some(8220440060596340118)), uuid(Some(3043154355299070039)), uuid(Some(782073527047025010)), uuid(Some(-4266509087205733116)), uuid(Some(5877939646495165431)), uuid(Some(-5313243819458142907)))), assignment('timestamp, unix_timestamp(current_timestamp(), yyyy-MM-dd HH:mm:ss, Some(UTC), false))], ('review_date = 1999-01-02)
+- 'UnresolvedRelation [reviews_mor_1], [], false

  at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.failAnalysis(package.scala:42)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:130)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1$adapted(CheckAnalysis.scala:1

In [18]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ COMMITS ]


[ COMPACTIONS ]


[ STORAGE ]

/home/emr-notebook/timeline
    0 used in 0 directories, 0 files


### 2.8. Update 258KB / +1 Log File

In [19]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-03'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.sql.AnalysisException: Table or view not found: reviews_mor_1; line 2 pos 4;
'UpdateTable [assignment('review_body, concat(uuid(Some(6472903392626216404)), uuid(Some(418195538071601691)), uuid(Some(765361974956442152)), uuid(Some(5404199400088367059)), uuid(Some(-1579440787788924911)), uuid(Some(1666527771465354179)), uuid(Some(2323369462597597570)), uuid(Some(3895849292314390834)), uuid(Some(616658989893851994)), uuid(Some(4331742821202858391)))), assignment('timestamp, unix_timestamp(current_timestamp(), yyyy-MM-dd HH:mm:ss, Some(UTC), false))], ('review_date = 1999-01-03)
+- 'UnresolvedRelation [reviews_mor_1], [], false

  at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.failAnalysis(package.scala:42)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:130)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1$adapted(CheckAnalysis.scala:101)
 

In [20]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ COMMITS ]


[ COMPACTIONS ]


[ STORAGE ]

/home/emr-notebook/timeline
    0 used in 0 directories, 0 files


### 2.9. Update 280KB / +1 Log File

In [21]:
%%sql
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-04'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
org.apache.spark.sql.AnalysisException: Table or view not found: reviews_mor_1; line 2 pos 4;
'UpdateTable [assignment('review_body, concat(uuid(Some(-5214990016055477583)), uuid(Some(7242138313244173318)), uuid(Some(-3253570886145918092)), uuid(Some(6755927654028793831)), uuid(Some(-4868479290752836557)), uuid(Some(8204372386663595752)), uuid(Some(1012849224109093689)), uuid(Some(3489432507892523572)), uuid(Some(8148638545384433885)), uuid(Some(-3093070761672460572)))), assignment('timestamp, unix_timestamp(current_timestamp(), yyyy-MM-dd HH:mm:ss, Some(UTC), false))], ('review_date = 1999-01-04)
+- 'UnresolvedRelation [reviews_mor_1], [], false

  at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.failAnalysis(package.scala:42)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:130)
  at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1$adapted(CheckAnalysis.scala:

In [22]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ COMMITS ]


[ COMPACTIONS ]


[ STORAGE ]

/home/emr-notebook/timeline
    0 used in 0 directories, 0 files


## 3. Test Case 2 - Async Compaction: Inline Schedule + Offline Execute

### 3.1. Create Table

In [23]:
%%sql
set TABLE_NAME=reviews_mor_2

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [24]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_2

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [25]:
%env TABLE_NAME=reviews_mor_2

env: TABLE_NAME=reviews_mor_2


In [26]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_2

env: TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_2


In [27]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 3

reviews_mor_2


In [28]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [29]:
%%sql
drop table if exists ${TABLE_NAME}_ro

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [30]:
%%sql
drop table if exists ${TABLE_NAME}_rt

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [31]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    hoodie.compact.inline = 'false',
    hoodie.compact.schedule.inline = 'true',
    hoodie.compact.inline.max.delta.commits = '3'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 3.2. Batch 1 - Insert

In [32]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [33]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230310101954805 │ deltacommit │ COMPLETED │ 03-10 10:19 │ 03-10 10:20 │ 03-10 10:20 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪════════

### 3.3. Batch 2 - Update

In [34]:
%%sql
-- 更新1999-01-01的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [35]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230310101954805 │ deltacommit │ COMPLETED │ 03-10 10:19 │ 03-10 10:20 │ 03-10 10:20 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230310102110571 │ deltacommit │ COMPLETED │ 03-10 10:21 │ 03-10 10:21 │ 03-10 10:21 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        

### 3.4. Batch 3 - Update

In [36]:
%%sql
-- 更新1999-01-02的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-02'
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [37]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230310101954805 │ deltacommit │ COMPLETED │ 03-10 10:19 │ 03-10 10:20 │ 03-10 10:20 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230310102110571 │ deltacommit │ COMPLETED │ 03-10 10:21 │ 03-10 10:21 │ 03-10 10:21 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230310102152303 │ deltacommit │ COMPLETED │ 03-10 10:21 │ 03-10 10:22 │ 03-10 10:22 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230310102209704

### 3.5. Async Execute Compaction

In [38]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class "org.apache.hudi.utilities.HoodieCompactor" \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" > ~/${TABLE_NAME}.compaction.execute.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage

## 4. Test Case 3 - Async Compaction: Offline Schedule + Offline Execute

### 4.1. Create Table

In [ ]:
%%sql
set TABLE_NAME=reviews_mor_3

In [ ]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_3

In [ ]:
%env TABLE_NAME=reviews_mor_3

In [ ]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_mor_3

In [ ]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 3

In [ ]:
%%sql
drop table if exists ${TABLE_NAME}

In [ ]:
%%sql
drop table if exists ${TABLE_NAME}_ro

In [ ]:
%%sql
drop table if exists ${TABLE_NAME}_rt

In [ ]:
%%sql
create table if not exists ${TABLE_NAME} (
    review_id string, 
    star_rating long, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175',
    -- hoodie.compact.inline = 'false',
    -- hoodie.compact.schedule.inline = 'false',
    hoodie.logfile.max.size = '512000',
    hoodie.compact.inline.max.delta.commits = '3'
);

### 4.2. Batch 1 - Insert

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1999

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage

### 4.3. Batch 2 - Update

In [ ]:
%%sql
-- 更新1999-01-01的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    review_date = '1999-01-01'
;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage

### 4.4. Batch 3 - Update

In [ ]:
%%sql
-- 更新1999-01-02的评价
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    star_rating = 5
;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage

### 4.5. Async Schedule Compaction

In [ ]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieCompactor' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'schedule' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.compact.inline.max.delta.commits=3" > ~/${TABLE_NAME}.compaction.schedule.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage

### 4.6. Async Execute Compaction

In [ ]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class "org.apache.hudi.utilities.HoodieCompactor" \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" > ~/${TABLE_NAME}.compaction.execute.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits compactions storage